In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'notebooks' / 'src' / 'models'

df = pd.read_parquet(str(DATA_DIR / 'jaipur_simulated_90d.parquet'))
df.head(2)

In [3]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)

df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
df['month_cos'] = np.cos(2 * np.pi * df['month']/12)

print(df[['hour','hour_sin', 'hour_cos']].head(25))

                           hour      hour_sin      hour_cos
2024-01-01 00:00:00+05:30     0  0.000000e+00  1.000000e+00
2024-01-01 01:00:00+05:30     1  2.588190e-01  9.659258e-01
2024-01-01 02:00:00+05:30     2  5.000000e-01  8.660254e-01
2024-01-01 03:00:00+05:30     3  7.071068e-01  7.071068e-01
2024-01-01 04:00:00+05:30     4  8.660254e-01  5.000000e-01
2024-01-01 05:00:00+05:30     5  9.659258e-01  2.588190e-01
2024-01-01 06:00:00+05:30     6  1.000000e+00  6.123234e-17
2024-01-01 07:00:00+05:30     7  9.659258e-01 -2.588190e-01
2024-01-01 08:00:00+05:30     8  8.660254e-01 -5.000000e-01
2024-01-01 09:00:00+05:30     9  7.071068e-01 -7.071068e-01
2024-01-01 10:00:00+05:30    10  5.000000e-01 -8.660254e-01
2024-01-01 11:00:00+05:30    11  2.588190e-01 -9.659258e-01
2024-01-01 12:00:00+05:30    12  1.224647e-16 -1.000000e+00
2024-01-01 13:00:00+05:30    13 -2.588190e-01 -9.659258e-01
2024-01-01 14:00:00+05:30    14 -5.000000e-01 -8.660254e-01
2024-01-01 15:00:00+05:30    15 -7.07106

In [4]:
df['solar_lag_1h'] = df['solar_output_mw'].shift(1)
df['solar_lag_24h'] = df['solar_output_mw'].shift(24)
df['solar_lag_48h'] = df['solar_output_mw'].shift(48)
df['solar_lag_168h'] = df['solar_output_mw'].shift(168)

print(df[['solar_output_mw','solar_lag_1h', 'solar_lag_24h']].head(30))


                           solar_output_mw  solar_lag_1h  solar_lag_24h
2024-01-01 00:00:00+05:30         0.000000           NaN            NaN
2024-01-01 01:00:00+05:30         0.000000      0.000000            NaN
2024-01-01 02:00:00+05:30         0.000000      0.000000            NaN
2024-01-01 03:00:00+05:30         0.000000      0.000000            NaN
2024-01-01 04:00:00+05:30         0.000000      0.000000            NaN
2024-01-01 05:00:00+05:30         0.000000      0.000000            NaN
2024-01-01 06:00:00+05:30         0.000000      0.000000            NaN
2024-01-01 07:00:00+05:30         6.797668      0.000000            NaN
2024-01-01 08:00:00+05:30        19.916920      6.797668            NaN
2024-01-01 09:00:00+05:30        31.590889     19.916920            NaN
2024-01-01 10:00:00+05:30        41.709538     31.590889            NaN
2024-01-01 11:00:00+05:30        47.556450     41.709538            NaN
2024-01-01 12:00:00+05:30        50.331215     47.556450        

In [5]:
df['solar_rolling_mean_3h'] = df['solar_output_mw'].rolling(window=3).mean()
df['solar_rolling_mean_6h'] = df['solar_output_mw'].rolling(window=6).mean()
df['solar_rolling_std_3h'] = df['solar_output_mw'].rolling(window=3).std()

print(df[['solar_output_mw', 
          'solar_rolling_mean_3h', 
          'solar_rolling_std_3h']].iloc[20:30])

                           solar_output_mw  solar_rolling_mean_3h  \
2024-01-01 20:00:00+05:30              0.0               2.283358   
2024-01-01 21:00:00+05:30              0.0               0.000000   
2024-01-01 22:00:00+05:30              0.0               0.000000   
2024-01-01 23:00:00+05:30              0.0               0.000000   
2024-01-02 00:00:00+05:30              0.0               0.000000   
2024-01-02 01:00:00+05:30              0.0               0.000000   
2024-01-02 02:00:00+05:30              0.0               0.000000   
2024-01-02 03:00:00+05:30              0.0               0.000000   
2024-01-02 04:00:00+05:30              0.0               0.000000   
2024-01-02 05:00:00+05:30              0.0               0.000000   

                           solar_rolling_std_3h  
2024-01-01 20:00:00+05:30              3.954892  
2024-01-01 21:00:00+05:30              0.000000  
2024-01-01 22:00:00+05:30              0.000000  
2024-01-01 23:00:00+05:30              0

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.pipeline import clear_sky_ghi
from src.utils.config_loader import get_config

# clear_sky_index REPLACES the old clear_sky_ratio, which divided
# solar_output_mw (the forecasting TARGET) by a same-timestamp feature —
# pure target leakage (see roadmap task P0.1 / audit finding F1).
#
# This version only ever reads shortwave_radiation (a weather variable,
# available from a forecast ahead of time) and a solar-position-only
# clear-sky estimate (pvlib's Ineichen model) — computable for a future
# timestamp with zero generation data.
config = get_config()
lat = config['location']['latitude']
lon = config['location']['longitude']
alt = config['location'].get('elevation_m', 0.0)

df['clear_sky_ghi_model'] = clear_sky_ghi(df.index, lat, lon, alt).to_numpy()

df['clear_sky_index'] = np.where(
    df['clear_sky_ghi_model'] > 1.0,
    df['shortwave_radiation'] / df['clear_sky_ghi_model'],
    0.0,
)
df['clear_sky_index'] = df['clear_sky_index'].clip(0, 1.3)

print(df[['shortwave_radiation', 'clear_sky_ghi_model', 'clear_sky_index']].describe())


In [7]:
df_features = df.dropna()

print("Features created:", df_features.shape[1], "columns")
print("Rows remaining:", len(df_features))
print("\nAll columns:")
for col in df_features.columns:
    print(f"  {col}")
    
print(df['solar_lag_168h'].isna().sum())

Features created: 27 columns
Rows remaining: 1992

All columns:
  temperature_2m
  relative_humidity_2m
  precipitation
  cloud_cover
  wind_speed_10m
  shortwave_radiation
  direct_radiation
  diffuse_radiation
  solar_output_mw
  clear_sky_ghi
  hour
  day_of_week
  month
  is_daytime
  hour_sin
  hour_cos
  month_sin
  month_cos
  solar_lag_1h
  solar_lag_24h
  solar_lag_48h
  solar_lag_168h
  solar_rolling_mean_3h
  solar_rolling_mean_6h
  solar_rolling_std_3h
  clear_sky_output
  clear_sky_ratio
168


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'notebooks' / 'src' / 'models'

df_features = df.dropna()
df_features.to_parquet(str(DATA_DIR / 'jaipur_features_90d.parquet'))

print('Saved Successfully ')
print('Shape:', df_features.shape)

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'notebooks' / 'src' / 'models'

df_check = pd.read_parquet(str(DATA_DIR / 'jaipur_features_90d.parquet'))
print("Reloaded shape:", df_check.shape)
print("Columns:", df_check.shape[1])